# 🚀 PIPELINE HUẤN LUYỆN TOÀN DIỆN VDU (VIETNAMESE RECEIPT OCR & KIE)
### Tự động chạy tuần tự qua đêm (Overnight Run) - Sáng mai chỉ việc tải Model về!
Quy trình gồm 7 bước tự động:
1. **Cài đặt thư viện**
2. **Train VietOCR** (Nhận diện chữ tiếng Việt có dấu) -> `transformerocr.pth`
3. **Tạo dữ liệu chuẩn hóa** (EasyOCR CRAFT + VietOCR Batch)
4. **Pseudo-Labeling mở rộng dữ liệu** từ ~1.000 lên ~2.200 ảnh từ tập Private Test
5. **Huấn luyện LayoutLMv3 nâng cao** (Full Sub-token BIO + Weighted Loss + BBox Augmentation)
6. **Xuất ONNX & Lượng tử hóa INT8** -> Đóng gói `vdu_production_weights.zip`
7. **Test suy luận thực tế** trên hóa đơn mẫu & vẽ bounding box


In [ ]:
# ==============================================================================
# BƯỚC 1: CÀI ĐẶT TẤT CẢ THƯ VIỆN CẦN THIẾT
# ==============================================================================
!pip install -q "pillow>=10.4.0" "numpy==1.26.4" "torchvision"
!python -m pip install -q --no-deps vietocr
!pip install -q einops gdown prefetch-generator imgaug albumentations lmdb
!pip install -q --upgrade transformers accelerate datasets seqeval evaluate easyocr
!pip install -q onnx onnxscript onnxruntime onnxruntime-tools
print("✅ Cài đặt thư viện hoàn tất!")


In [ ]:
# ==============================================================================
# BƯỚC 2: CẮT DÒNG & HUẤN LUYỆN VIETOCR (TẠO transformerocr.pth)
# ==============================================================================
import ast, os, random, shutil, cv2, torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from vietocr.tool.config import Cfg
from vietocr.model.trainer import Trainer

DATA_DIR = "/kaggle/input/datasets/tuananh35/mc-ocr-2021/mcocr_public_train_test_shared_data/mcocr_public_train_test_shared_data/mcocr_train_data"
CSV_PATH = os.path.join(DATA_DIR, "mcocr_train_df.csv")
IMAGES_DIR = os.path.join(DATA_DIR, "train_images")

OUTPUT_CROP_DIR = "/kaggle/working/vietocr_data/train_crops"
TRAIN_TXT = "/kaggle/working/vietocr_data/train.txt"
VAL_TXT = "/kaggle/working/vietocr_data/val.txt"
WEIGHTS_DIR = "/kaggle/working/weights"

os.makedirs(OUTPUT_CROP_DIR, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# 1. Cắt ảnh từng dòng chữ
df = pd.read_csv(CSV_PATH)
gt_lines = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="1/2 Cắt dòng cho VietOCR"):
    img_name = str(row["img_id"]).strip()
    if not img_name.endswith((".jpg", ".png", ".jpeg")):
        img_name += ".jpg"
    img_path = os.path.join(IMAGES_DIR, img_name)
    if not os.path.exists(img_path) or pd.isna(row.get("anno_polygons")) or pd.isna(row.get("anno_texts")):
        continue

    img = cv2.imread(img_path)
    if img is None:
        continue
    h, w = img.shape[:2]

    texts = str(row["anno_texts"]).split("|||")
    try:
        poly_data = ast.literal_eval(row["anno_polygons"])
    except Exception:
        continue

    for line_idx in range(min(len(texts), len(poly_data))):
        text = texts[line_idx].strip()
        bbox = poly_data[line_idx].get("bbox")
        if not text or not bbox or len(bbox) < 4:
            continue
        bx, by, bw, bh = [int(v) for v in bbox]
        x_min, y_min = max(0, bx), max(0, by)
        x_max, y_max = min(w, bx + bw), min(h, by + bh)
        if (x_max - x_min < 5) or (y_max - y_min < 5):
            continue

        crop_name = f"{img_name.rsplit(".", 1)[0]}_{line_idx}.jpg"
        cv2.imwrite(os.path.join(OUTPUT_CROP_DIR, crop_name), img[y_min:y_max, x_min:x_max])
        gt_lines.append(f"{crop_name}\t{text}\n")

random.seed(42)
random.shuffle(gt_lines)
split = int(0.9 * len(gt_lines))

with open(TRAIN_TXT, "w", encoding="utf-8") as f:
    f.writelines(gt_lines[:split])
with open(VAL_TXT, "w", encoding="utf-8") as f:
    f.writelines(gt_lines[split:])

print(f"✅ Đã chuẩn bị {len(gt_lines)} dòng chữ (Train: {split} | Val: {len(gt_lines)-split})")

# 2. Huấn luyện VietOCR
for c_dir in ["train_receipt_ocr", "valid_receipt_ocr"]:
    if os.path.exists(c_dir):
        shutil.rmtree(c_dir)

class CustomVietOCRTrainer(Trainer):
    def validate(self):
        self.model.eval()
        total_loss = []
        with torch.no_grad():
            for step, batch in enumerate(self.valid_gen):
                batch = self.batch_to_device(batch)
                output = self.model(batch["img"], batch["tgt_input"])
                loss = self.criterion(output.flatten(0, 1), batch["tgt_output"].flatten())
                total_loss.append(loss.item())
                if step >= 50:
                    break
        self.model.train()
        return np.mean(total_loss) if len(total_loss) > 0 else 0.0

    def predict(self, sample=None):
        self.model.eval()
        pred_sents, actual_sents, img_files = [], [], []
        prob = 0.0
        with torch.no_grad():
            for batch in self.valid_gen:
                batch = self.batch_to_device(batch)
                translated, prob = self.model.predict(batch["img"])
                pred_sents.extend(translated)
                actual_sents.extend(self.vocab.batch_decode(batch["tgt_input"].tolist()))
                if sample and len(pred_sents) >= sample:
                    break
        self.model.train()
        return pred_sents, actual_sents, img_files, prob

config = Cfg.load_config_from_name("vgg_transformer")
config["dataset"].update({
    "name": "receipt_ocr",
    "data_root": OUTPUT_CROP_DIR,
    "train_annotation": TRAIN_TXT,
    "valid_annotation": VAL_TXT,
    "image_height": 32,
    "image_min_width": 32,
    "image_max_width": 512
})
config["trainer"].update({
    "print_every": 100,
    "valid_every": 200,
    "iters": 1000,
    "checkpoint": os.path.join(WEIGHTS_DIR, "checkpoint.pth"),
    "export": os.path.join(WEIGHTS_DIR, "transformerocr.pth"),
    "batch_size": 64,
    "metrics": 200
})
config["optimizer"]["max_lr"] = 1e-4
config["device"] = "cuda:0" if torch.cuda.is_available() else "cpu"

print("🔥 Bắt đầu huấn luyện VietOCR (1000 iters)...")
trainer = CustomVietOCRTrainer(config, pretrained=True)
trainer.train()
print("✅ Đã lưu VietOCR tại: /kaggle/working/weights/transformerocr.pth")


In [ ]:
# ==============================================================================
# BƯỚC 3: TẠO DỮ LIỆU ĐỒNG NHẤT CHO LAYOUTLMV3 (EASYOCR + VIETOCR BATCH)
# ==============================================================================
import ast, json, os, warnings, cv2, easyocr, torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image
from vietocr.tool.config import Cfg
from vietocr.tool.predictor import Predictor

warnings.filterwarnings("ignore", category=RuntimeWarning)
device = "cuda:0" if torch.cuda.is_available() else "cpu"

CSV_PATH = "/kaggle/input/datasets/tuananh35/mc-ocr-2021/mcocr_public_train_test_shared_data/mcocr_public_train_test_shared_data/mcocr_train_data/mcocr_train_df.csv"
IMG_DIR = "/kaggle/input/datasets/tuananh35/mc-ocr-2021/mcocr_public_train_test_shared_data/mcocr_public_train_test_shared_data/mcocr_train_data/train_images"
OUTPUT_JSON = "/kaggle/working/layoutlm_train_data.json"

detector = easyocr.Reader(["en"], gpu=torch.cuda.is_available(), recognizer=False)

# Dùng mô hình vgg_transformer vừa train ở Bước 2
v_cfg = Cfg.load_config_from_name("vgg_transformer")
custom_weight = "/kaggle/working/weights/transformerocr.pth"
if os.path.exists(custom_weight):
    v_cfg["weights"] = custom_weight
v_cfg["device"] = device
v_cfg["predictor"]["beamsearch"] = False
vietocr_reader = Predictor(v_cfg)

LABEL_MAP = {
    "SELLER": "MERCHANT_NAME",
    "ADDRESS": "ADDRESS",
    "TIMESTAMP": "DATE",
    "TOTAL_COST": "TOTAL_AMOUNT",
}

def compute_iou(boxA, boxB):
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return inter / float(areaA + areaB - inter + 1e-6)

def norm_box(b, w, h):
    return [
        int(max(0, min(1000, (b[0] / w) * 1000))),
        int(max(0, min(1000, (b[1] / h) * 1000))),
        int(max(0, min(1000, (b[2] / w) * 1000))),
        int(max(0, min(1000, (b[3] / h) * 1000))),
    ]

def sort_boxes_reading_order(boxes, words, tags):
    if not boxes:
        return [], [], []
    combined = list(zip(boxes, words, tags))
    heights = [(b[3] - b[1]) for b in boxes]
    y_threshold = max((np.median(heights) if heights else 20) * 0.5, 10)
    combined_sorted_y = sorted(combined, key=lambda item: (item[0][1] + item[0][3]) / 2)
    lines, cur_line = [], [combined_sorted_y[0]]
    for item in combined_sorted_y[1:]:
        item_cy = (item[0][1] + item[0][3]) / 2
        line_cy = np.mean([(x[0][1] + x[0][3]) / 2 for x in cur_line])
        if abs(item_cy - line_cy) <= y_threshold:
            cur_line.append(item)
        else:
            lines.append(sorted(cur_line, key=lambda x: x[0][0]))
            cur_line = [item]
    if cur_line:
        lines.append(sorted(cur_line, key=lambda x: x[0][0]))
    flat = [item for line in lines for item in line]
    return [f[0] for f in flat], [f[1] for f in flat], [f[2] for f in flat]

df = pd.read_csv(CSV_PATH)
dataset_with_o = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Chuẩn bị dữ liệu LayoutLMv3 (Batch)"):
    img_id = str(row["img_id"]).strip()
    img_path = os.path.join(IMG_DIR, img_id)
    if not os.path.exists(img_path) or pd.isna(row["anno_texts"]):
        continue

    img = cv2.imread(img_path)
    if img is None:
        continue
    h, w = img.shape[:2]
    img_pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    try:
        annos = ast.literal_eval(row["anno_polygons"])
    except Exception:
        continue
    texts = str(row["anno_texts"]).split("|||")
    labels = str(row["anno_labels"]).split("|||") if not pd.isna(row.get("anno_labels")) else []

    all_boxes, all_words, all_tags = [], [], []
    for i in range(min(len(annos), len(texts))):
        txt = texts[i].strip()
        bbox = annos[i].get("bbox")
        if not bbox or not txt:
            continue
        bx, by, bw, bh = [int(v) for v in bbox]
        raw_box = [max(0, bx), max(0, by), min(w, bx + bw), min(h, by + bh)]
        if raw_box[2] > raw_box[0] and raw_box[3] > raw_box[1]:
            all_boxes.append(raw_box)
            all_words.append(txt)
            all_tags.append(LABEL_MAP.get(labels[i].strip() if i < len(labels) else "O", "O"))

    horiz_boxes, _ = detector.detect(img, canvas_size=1024)
    if horiz_boxes and len(horiz_boxes[0]) > 0:
        o_boxes, o_crops = [], []
        for b in horiz_boxes[0]:
            ocr_box = [max(0, int(b[0])), max(0, int(b[2])), min(w, int(b[1])), min(h, int(b[3]))]
            if (ocr_box[2] - ocr_box[0] < 10) or (ocr_box[3] - ocr_box[1] < 10):
                continue
            if not any(compute_iou(ocr_box, gt) > 0.25 for gt in all_boxes):
                o_boxes.append(ocr_box)
                o_crops.append(img_pil.crop((ocr_box[0], ocr_box[1], ocr_box[2], ocr_box[3])))

        if o_crops:
            pred_texts = vietocr_reader.predict_batch(o_crops)
            for ocr_box, text_o in zip(o_boxes, pred_texts):
                text_o = str(text_o).strip()
                if text_o:
                    all_boxes.append(ocr_box)
                    all_words.append(text_o)
                    all_tags.append("O")

    if not all_boxes:
        continue

    s_boxes, s_words, s_tags = sort_boxes_reading_order(all_boxes, all_words, all_tags)
    dataset_with_o.append({
        "id": img_id,
        "image_path": img_path,
        "words": s_words,
        "bboxes": [norm_box(b, w, h) for b in s_boxes],
        "ner_tags": s_tags,
    })

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(dataset_with_o, f, ensure_ascii=False, indent=2)
print(f"✅ Đã tạo thành công {len(dataset_with_o)} mẫu chuẩn tại: {OUTPUT_JSON}")


In [ ]:
# ==============================================================================
# BƯỚC 4: TỰ ĐỘNG MỞ RỘNG DỮ LIỆU LÊN ~2.200 ẢNH (PSEUDO-LABELING TRÊN PRIVATE TEST)
# ==============================================================================
import gc, glob, json, os, random, cv2, torch, evaluate
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict
from PIL import Image
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoProcessor,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
)

# 1. Huấn luyện LayoutLMv3 sơ bộ Phase 1 (8 epochs ~3 phút) trên tập gốc
print("🔥 Huấn luyện LayoutLMv3 Phase 1 để chuẩn bị Pseudo-labeling...")
with open("/kaggle/working/layoutlm_train_data.json", "r", encoding="utf-8") as f:
    base_data = json.load(f)

LABELS = [
    "O",
    "B-MERCHANT_NAME", "I-MERCHANT_NAME",
    "B-ADDRESS", "I-ADDRESS",
    "B-DATE", "I-DATE",
    "B-TOTAL_AMOUNT", "I-TOTAL_AMOUNT"
]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}

def convert_to_bio(ner_tags):
    bio_tags, prev = [], None
    for tag in ner_tags:
        if tag == "O" or tag not in ["MERCHANT_NAME", "ADDRESS", "DATE", "TOTAL_AMOUNT"]:
            bio_tags.append("O")
            prev = None
        elif tag == prev:
            bio_tags.append(f"I-{tag}")
        else:
            bio_tags.append(f"B-{tag}")
            prev = tag
    return bio_tags

samples_p1 = []
for item in base_data:
    words, bboxes, ner_tags = item.get("words", []), item.get("bboxes", []), item.get("ner_tags", [])
    if words and len(words) == len(bboxes) == len(ner_tags) and os.path.exists(item["image_path"]):
        samples_p1.append({
            "image_path": item["image_path"],
            "tokens": words,
            "bboxes": bboxes,
            "ner_tags": [label2id[t] for t in convert_to_bio(ner_tags)]
        })

MODEL_CHECKPOINT = "microsoft/layoutlmv3-base-chinese"
processor = AutoProcessor.from_pretrained(MODEL_CHECKPOINT, apply_ocr=False)
MAX_LEN = 256

def prepare_fn(ex):
    image = Image.open(ex["image_path"]).convert("RGB")
    tokenized = processor.tokenizer(
        ex["tokens"], truncation=True, padding="max_length", max_length=MAX_LEN,
        is_split_into_words=True, return_tensors="pt"
    )
    word_ids = tokenized.word_ids(batch_index=0)
    aligned_b, aligned_l = [], []
    prev_w = None
    for w in word_ids:
        if w is None:
            aligned_b.append([0, 0, 0, 0])
            aligned_l.append(-100)
        elif w != prev_w:
            aligned_b.append(ex["bboxes"][w])
            aligned_l.append(ex["ner_tags"][w])
        else:
            aligned_b.append(ex["bboxes"][w])
            orig = id2label[ex["ner_tags"][w]]
            aligned_l.append(label2id["I-" + orig[2:]] if orig.startswith("B-") else ex["ner_tags"][w])
        prev_w = w

    return {
        "input_ids": tokenized["input_ids"].squeeze(0),
        "attention_mask": tokenized["attention_mask"].squeeze(0),
        "bbox": torch.tensor(aligned_b, dtype=torch.long),
        "labels": torch.tensor(aligned_l, dtype=torch.long),
        "pixel_values": processor.image_processor(images=image, return_tensors="pt")["pixel_values"].squeeze(0)
    }

ds_p1 = Dataset.from_list(samples_p1).map(prepare_fn, remove_columns=["image_path", "tokens", "bboxes", "ner_tags"])
ds_p1.set_format("torch")

model_p1 = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=len(LABELS), id2label=id2label, label2id=label2id
)

trainer_p1 = Trainer(
    model=model_p1,
    args=TrainingArguments(
        output_dir="/tmp/p1_checkpoints",
        num_train_epochs=8,
        per_device_train_batch_size=4,
        learning_rate=4e-5,
        fp16=True,
        logging_steps=50,
        report_to="none"
    ),
    train_dataset=ds_p1,
)
trainer_p1.train()
model_p1.save_pretrained("/tmp/layoutlm_p1")
print("✅ Huấn luyện Phase 1 hoàn tất! Bắt đầu Pseudo-labeling trên Private Test...")

# 2. Pseudo-labeling trên Private Test
PRIVATE_TEST_DIR = "/kaggle/input/datasets/tuananh35/mc-ocr-2021/mcocr_private_test_data/mcocr_private_test_data/test_images"
AUGMENTED_JSON = "/kaggle/working/layoutlm_train_augmented.json"

if os.path.exists(PRIVATE_TEST_DIR):
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    kie_model = AutoModelForTokenClassification.from_pretrained("/tmp/layoutlm_p1").to(device).eval()
    test_images = sorted(glob.glob(os.path.join(PRIVATE_TEST_DIR, "*.jpg")))

    added_count = 0
    augmented_dataset = list(base_data)

    for img_path in tqdm(test_images, desc="Pseudo-labeling"):
        img_cv = cv2.imread(img_path)
        if img_cv is None:
            continue
        h, w = img_cv.shape[:2]
        img_pil = Image.fromarray(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))

        horiz_boxes, _ = detector.detect(img_cv, canvas_size=1024)
        if not horiz_boxes or len(horiz_boxes[0]) == 0:
            continue

        cand_b, cand_c = [], []
        for b in horiz_boxes[0]:
            x1, x2, y1, y2 = max(0, int(b[0])), min(w, int(b[1])), max(0, int(b[2])), min(h, int(b[3]))
            if x2 - x1 < 10 or y2 - y1 < 10:
                continue
            cand_b.append([x1, y1, x2, y2])
            cand_c.append(img_pil.crop((x1, y1, x2, y2)))

        if len(cand_c) < 5:
            continue

        pred_txts = vietocr_reader.predict_batch(cand_c)
        raw_b, raw_w = [], []
        for box, txt in zip(cand_b, pred_txts):
            txt = str(txt).strip()
            if txt:
                raw_b.append(box)
                raw_w.append(txt)

        if len(raw_w) < 5:
            continue

        s_b, s_w, _ = sort_boxes_reading_order(raw_b, raw_w, ["O"] * len(raw_w))
        norm_b = [norm_box(b, w, h) for b in s_b]

        tokenized = processor.tokenizer(s_w, is_split_into_words=True, truncation=True, padding="max_length", max_length=MAX_LEN)
        word_ids = tokenized.word_ids()
        aligned_b = [norm_b[wi] if wi is not None else [0, 0, 0, 0] for wi in word_ids]

        with torch.no_grad():
            out = kie_model(
                input_ids=torch.tensor([tokenized["input_ids"]], dtype=torch.long, device=device),
                attention_mask=torch.tensor([tokenized["attention_mask"]], dtype=torch.long, device=device),
                bbox=torch.tensor([aligned_b], dtype=torch.long, device=device),
                pixel_values=processor.image_processor(images=img_pil, return_tensors="pt")["pixel_values"].to(device),
            )
            probs = F.softmax(out.logits, dim=2)[0].cpu().numpy()
            preds = np.argmax(probs, axis=1)

        sub_tags, sub_confs = defaultdict(list), defaultdict(list)
        for idx, wi in enumerate(word_ids):
            if wi is not None:
                tag = id2label[int(preds[idx])].replace("B-", "").replace("I-", "")
                sub_tags[wi].append(tag)
                sub_confs[wi].append(float(probs[idx, preds[idx]]))

        p_tags, p_confs = [], []
        for i in range(len(s_w)):
            t_i = sub_tags.get(i, ["O"])
            non_o = [t for t in t_i if t != "O"]
            tag_f = max(set(non_o), key=non_o.count) if non_o else "O"
            p_tags.append(tag_f)
            if tag_f != "O":
                p_confs.append(np.mean(sub_confs[i]))

        if p_confs and np.mean(p_confs) >= 0.88 and "MERCHANT_NAME" in p_tags and "TOTAL_AMOUNT" in p_tags:
            augmented_dataset.append({
                "id": os.path.basename(img_path),
                "image_path": img_path,
                "words": s_w,
                "bboxes": norm_b,
                "ner_tags": p_tags,
            })
            added_count += 1

    with open(AUGMENTED_JSON, "w", encoding="utf-8") as f:
        json.dump(augmented_dataset, f, ensure_ascii=False, indent=2)
    print(f"🎉 Đã mở rộng thêm {added_count} ảnh! Tổng dataset mới: {len(augmented_dataset)} ảnh.")
else:
    shutil.copy("/kaggle/working/layoutlm_train_data.json", AUGMENTED_JSON)
    print("ℹ️ Dùng tập train gốc làm dữ liệu cuối cùng.")


In [ ]:
# ==============================================================================
# BƯỚC 5: HUẤN LUYỆN TOÀN DIỆN LAYOUTLMV3 TRÊN TOÀN BỘ ~2.200 ẢNH
# ==============================================================================
import gc, os, json, random, torch, evaluate
import torch.nn as nn
import numpy as np
from PIL import Image
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoProcessor,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
)

# 1. Dọn dẹp VRAM
for var_name in ["trainer_p1", "model_p1", "kie_model", "vietocr_reader", "detector"]:
    if var_name in globals():
        try:
            del globals()[var_name]
        except Exception:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

DATA_PATH = "/kaggle/working/layoutlm_train_augmented.json"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "/kaggle/working/layoutlm_train_data.json"

print(f"📂 Đang nạp dữ liệu huấn luyện cuối cùng từ: {DATA_PATH}")
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

LABELS = [
    "O",
    "B-MERCHANT_NAME", "I-MERCHANT_NAME",
    "B-ADDRESS", "I-ADDRESS",
    "B-DATE", "I-DATE",
    "B-TOTAL_AMOUNT", "I-TOTAL_AMOUNT"
]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}

def convert_to_bio(ner_tags):
    bio_tags, prev = [], None
    for tag in ner_tags:
        if tag == "O" or tag not in ["MERCHANT_NAME", "ADDRESS", "DATE", "TOTAL_AMOUNT"]:
            bio_tags.append("O")
            prev = None
        elif tag == prev:
            bio_tags.append(f"I-{tag}")
        else:
            bio_tags.append(f"B-{tag}")
            prev = tag
    return bio_tags

processed_samples = []
for item in raw_data:
    words, bboxes, ner_tags = item.get("words", []), item.get("bboxes", []), item.get("ner_tags", [])
    img_path = item.get("image_path", "")
    if words and len(words) == len(bboxes) == len(ner_tags) and os.path.exists(img_path):
        clean_w, clean_b, clean_t = [], [], []
        for w, b, t in zip(words, bboxes, ner_tags):
            w_str = str(w).strip()
            if w_str and len(b) == 4 and b[2] > b[0] and b[3] > b[1]:
                clean_w.append(w_str)
                clean_b.append([max(0, min(1000, int(c))) for c in b])
                clean_t.append(t)
        if clean_w:
            processed_samples.append({
                "id": item["id"],
                "image_path": img_path,
                "tokens": clean_w,
                "bboxes": clean_b,
                "ner_tags": [label2id[t] for t in convert_to_bio(clean_t)]
            })

print(f"✅ Đưa vào huấn luyện cuối cùng: {len(processed_samples)} ảnh!")
train_data, val_data = train_test_split(processed_samples, test_size=0.10, random_state=42)

MODEL_CHECKPOINT = "microsoft/layoutlmv3-base-chinese"
processor = AutoProcessor.from_pretrained(MODEL_CHECKPOINT, apply_ocr=False)
MAX_LEN = 256

def prepare_fn(ex):
    image = Image.open(ex["image_path"]).convert("RGB")
    tokenized = processor.tokenizer(
        ex["tokens"], truncation=True, padding="max_length", max_length=MAX_LEN,
        is_split_into_words=True, return_tensors="pt"
    )
    word_ids = tokenized.word_ids(batch_index=0)
    aligned_b, aligned_l = [], []
    prev_w = None
    for w in word_ids:
        if w is None:
            aligned_b.append([0, 0, 0, 0])
            aligned_l.append(-100)
        elif w != prev_w:
            aligned_b.append(ex["bboxes"][w])
            aligned_l.append(ex["ner_tags"][w])
        else:
            aligned_b.append(ex["bboxes"][w])
            orig = id2label[ex["ner_tags"][w]]
            aligned_l.append(label2id["I-" + orig[2:]] if orig.startswith("B-") else ex["ner_tags"][w])
        prev_w = w

    return {
        "input_ids": tokenized["input_ids"].squeeze(0),
        "attention_mask": tokenized["attention_mask"].squeeze(0),
        "bbox": torch.tensor(aligned_b, dtype=torch.long),
        "labels": torch.tensor(aligned_l, dtype=torch.long),
        "pixel_values": processor.image_processor(images=image, return_tensors="pt")["pixel_values"].squeeze(0)
    }

print("⚡ Đang nạp toàn bộ dataset vào RAM (~45 giây)...")
train_ds = Dataset.from_list(train_data).map(prepare_fn, remove_columns=["id", "image_path", "tokens", "bboxes", "ner_tags"])
val_ds = Dataset.from_list(val_data).map(prepare_fn, remove_columns=["id", "image_path", "tokens", "bboxes", "ner_tags"])
train_ds.set_format("torch")
val_ds.set_format("torch")

# BBox Augmentation trực tiếp trên Tensor
def fast_collator(features):
    batch = {
        "input_ids": torch.stack([f["input_ids"] for f in features]),
        "attention_mask": torch.stack([f["attention_mask"] for f in features]),
        "bbox": torch.stack([f["bbox"] for f in features]),
        "labels": torch.stack([f["labels"] for f in features]),
        "pixel_values": torch.stack([f["pixel_values"] for f in features]),
    }
    if random.random() < 0.5:
        noise = torch.randint(-8, 9, batch["bbox"].shape)
        mask = (batch["labels"] != -100).unsqueeze(-1)
        jittered = torch.clamp(batch["bbox"] + noise * mask, 0, 1000)
        valid = (jittered[:, :, 2] > jittered[:, :, 0]) & (jittered[:, :, 3] > jittered[:, :, 1])
        batch["bbox"] = torch.where(valid.unsqueeze(-1), jittered, batch["bbox"])
    return batch

class WeightedLayoutTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        class_weights = torch.tensor(
            [1.0, 4.0, 4.0, 4.0, 4.0, 5.0, 5.0, 5.0, 5.0],
            device=logits.device, dtype=logits.dtype
        )
        loss_fct = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)
        loss = loss_fct(logits.view(-1, logits.shape[-1]), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

metric = evaluate.load("seqeval")
def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)
    true_preds = [[LABELS[p] for (p, l) in zip(pred, lab) if l != -100] for pred, lab in zip(preds, labels)]
    true_labs = [[LABELS[l] for (p, l) in zip(pred, lab) if l != -100] for pred, lab in zip(preds, labels)]
    res = metric.compute(predictions=true_preds, references=true_labs)
    return {
        "precision": res["overall_precision"],
        "recall": res["overall_recall"],
        "f1": res["overall_f1"],
    }

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=len(LABELS), id2label=id2label, label2id=label2id
)

training_args = TrainingArguments(
    output_dir="/kaggle/working/layoutlmv3_final_checkpoints",
    num_train_epochs=16,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=4e-5,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    dataloader_num_workers=2,
    logging_steps=25,
    report_to="none"
)

trainer = WeightedLayoutTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=processor,
    data_collator=fast_collator,
    compute_metrics=compute_metrics,
)

print("🚀 Bắt đầu huấn luyện LayoutLMv3 chính thức (16 epochs)...")
trainer.train()

FINAL_DIR = "/kaggle/working/layoutlmv3_finetuned"
model.save_pretrained(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
print(f"✅ Đã lưu mô hình hoàn thiện tại: {FINAL_DIR}")


In [ ]:
# ==============================================================================
# BƯỚC 6: XUẤT ONNX INT8 & ĐÓNG GÓI vdu_production_weights.zip
# ==============================================================================
import os, shutil, zipfile, torch
from transformers import AutoModelForTokenClassification, AutoProcessor
from onnxruntime.quantization import quantize_dynamic, QuantType

MODEL_DIR = "/kaggle/working/layoutlmv3_finetuned"
DEPLOY_DIR = "/kaggle/working/vdu_deployment"
ONNX_DIR = os.path.join(DEPLOY_DIR, "onnx_models")
WEIGHTS_DEPLOY_DIR = os.path.join(DEPLOY_DIR, "weights")
LAYOUTLM_DEPLOY_DIR = os.path.join(WEIGHTS_DEPLOY_DIR, "layoutlmv3_finetuned")

os.makedirs(ONNX_DIR, exist_ok=True)
os.makedirs(LAYOUTLM_DEPLOY_DIR, exist_ok=True)

fp32_onnx = "/kaggle/working/layoutlmv3_fp32.onnx"
int8_onnx = os.path.join(ONNX_DIR, "layoutlmv3_int8.onnx")

print("1/3 Xuất PyTorch model sang ONNX opset 14...")
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR).cpu().eval()
processor = AutoProcessor.from_pretrained(MODEL_DIR, apply_ocr=False)

processor.save_pretrained(LAYOUTLM_DEPLOY_DIR)
model.config.save_pretrained(LAYOUTLM_DEPLOY_DIR)

dummy_inputs = (
    torch.randint(0, 1000, (1, 256), dtype=torch.long),
    torch.randint(0, 500, (1, 256, 4), dtype=torch.long),
    torch.randn(1, 3, 224, 224, dtype=torch.float32),
    torch.ones((1, 256), dtype=torch.long),
)

torch.onnx.export(
    model,
    dummy_inputs,
    fp32_onnx,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=["input_ids", "bbox", "pixel_values", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "bbox": {0: "batch_size", 1: "sequence_length"},
        "pixel_values": {0: "batch_size"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "logits": {0: "batch_size", 1: "sequence_length"},
    },
    dynamo=False,
)

print("2/3 Lượng tử hóa động sang INT8...")
quantize_dynamic(
    model_input=fp32_onnx,
    model_output=int8_onnx,
    weight_type=QuantType.QInt8,
    op_types_to_quantize=["MatMul", "Gemm"],
)

# Chép trọng số VietOCR vào gói deployment
vietocr_p = "/kaggle/working/weights/transformerocr.pth"
if os.path.exists(vietocr_p):
    shutil.copy(vietocr_p, os.path.join(WEIGHTS_DEPLOY_DIR, "transformerocr.pth"))

print("3/3 Đóng gói toàn bộ vào vdu_production_weights.zip...")
zip_path = "/kaggle/working/vdu_production_weights.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(DEPLOY_DIR):
        for file in files:
            full_p = os.path.join(root, file)
            rel_p = os.path.relpath(full_p, DEPLOY_DIR)
            zipf.write(full_p, arcname=rel_p)

# Xóa file tạm
if os.path.exists(fp32_onnx):
    os.remove(fp32_onnx)

print("==================================================================")
print("🎉 THÀNH CÔNG RỰC RỠ! TOÀN BỘ QUÁ TRÌNH HUẤN LUYỆN ĐÃ XONG!")
print(f"📦 File kết quả: {zip_path} ({os.path.getsize(zip_path)/1024**2:.1f} MB)")
print("👉 Sáng mai bạn chỉ việc tải file vdu_production_weights.zip về giải nén vào máy!")
print("==================================================================")


In [ ]:
# ==============================================================================
# BƯỚC 7: KIỂM THỬ TRỰC QUAN TRÊN HÓA ĐƠN THỰC TẾ
# ==============================================================================
import glob, json, os, re, cv2, easyocr, torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoProcessor
import onnxruntime as ort
from vietocr.tool.config import Cfg
from vietocr.tool.predictor import Predictor

TEST_DIR = "/kaggle/input/datasets/tuananh35/mc-ocr-2021/mcocr_private_test_data/mcocr_private_test_data/test_images"
test_imgs = glob.glob(os.path.join(TEST_DIR, "*.jpg"))
sample_img = test_imgs[0] if test_imgs else None

if sample_img and os.path.exists("/kaggle/working/vdu_deployment/onnx_models/layoutlmv3_int8.onnx"):
    print(f"🔍 Đang kiểm thử trên ảnh: {sample_img}")
    img_cv = cv2.imread(sample_img)
    h, w = img_cv.shape[:2]
    img_draw = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_draw)

    detector = easyocr.Reader(["en"], gpu=torch.cuda.is_available(), recognizer=False)
    v_cfg = Cfg.load_config_from_name("vgg_transformer")
    v_cfg["weights"] = "/kaggle/working/vdu_deployment/weights/transformerocr.pth"
    v_cfg["device"] = "cuda:0" if torch.cuda.is_available() else "cpu"
    ocr = Predictor(v_cfg)

    horiz_boxes, _ = detector.detect(img_cv, canvas_size=1024)
    raw_b, crops = [], []
    for b in horiz_boxes[0]:
        x1, x2, y1, y2 = max(0, int(b[0])), min(w, int(b[1])), max(0, int(b[2])), min(h, int(b[3]))
        if x2 - x1 >= 10 and y2 - y1 >= 10:
            raw_b.append([x1, y1, x2, y2])
            crops.append(img_pil.crop((x1, y1, x2, y2)))

    texts = ocr.predict_batch(crops)
    s_b, s_w, _ = sort_boxes_reading_order(raw_b, texts, ["O"] * len(texts))
    norm_b = [norm_box(b, w, h) for b in s_b]

    proc = AutoProcessor.from_pretrained("/kaggle/working/vdu_deployment/weights/layoutlmv3_finetuned", apply_ocr=False)
    tok = proc.tokenizer(s_w, is_split_into_words=True, truncation=True, padding="max_length", max_length=256, return_tensors="np")
    word_ids = tok.word_ids()
    aligned_b = np.array([norm_b[wi] if wi is not None else [0, 0, 0, 0] for wi in word_ids], dtype=np.int64)[None, :]
    pixel_vals = proc.image_processor(images=img_pil, return_tensors="np")["pixel_values"]

    session = ort.InferenceSession("/kaggle/working/vdu_deployment/onnx_models/layoutlmv3_int8.onnx", providers=["CPUExecutionProvider"])
    ort_inputs = {
        "input_ids": tok["input_ids"].astype(np.int64),
        "bbox": aligned_b,
        "pixel_values": pixel_vals.astype(np.float32),
        "attention_mask": tok["attention_mask"].astype(np.int64)
    }
    logits = session.run(["logits"], ort_inputs)[0]
    preds = np.argmax(logits, axis=2)[0]

    LABELS = ["O", "B-MERCHANT_NAME", "I-MERCHANT_NAME", "B-ADDRESS", "I-ADDRESS", "B-DATE", "I-DATE", "B-TOTAL_AMOUNT", "I-TOTAL_AMOUNT"]
    id2l = {i: l for i, l in enumerate(LABELS)}
    extracted = {"MERCHANT_NAME": [], "ADDRESS": [], "DATE": [], "TOTAL_AMOUNT": []}

    for idx, wi in enumerate(word_ids):
        if wi is not None:
            tag = id2l[preds[idx]].replace("B-", "").replace("I-", "")
            if tag in extracted and s_w[wi] not in extracted[tag]:
                extracted[tag].append(s_w[wi])

    print("📊 KẾT QUẢ BÓC TÁCH MÔ HÌNH INT8:")
    print(json.dumps({k: " ".join(v) for k, v in extracted.items()}, ensure_ascii=False, indent=2))
